# SILVER LAYER

## DATA LOADING

In [0]:
df_cal = (spark.read.format("csv")
         .option("header", True)
         .option("inferSchema", True)
         .load("abfss://bronze@sampeldatalake.dfs.core.windows.net/AdventureWorks_Calendar"))
display(df_cal)

Date
2015-01-01
2015-01-02
2015-01-03
2015-01-04
2015-01-05
2015-01-06
2015-01-07
2015-01-08
2015-01-09
2015-01-10


In [0]:
df_cust = (spark.read.format("csv")
         .option("header", True)
         .option("inferSchema", True)
         .load("abfss://bronze@sampeldatalake.dfs.core.windows.net/AdventureWorks_Customers"))

In [0]:
df_prod_catg = (spark.read.format("csv")
         .option("header", True)
         .option("inferSchema", True)
         .load("abfss://bronze@sampeldatalake.dfs.core.windows.net/AdventureWorks_Product_Categories"))

In [0]:
df_prod_sub = (spark.read.format("csv")
         .option("header", True)
         .option("inferSchema", True)
         .load("abfss://bronze@sampeldatalake.dfs.core.windows.net/AdventureWorks_Product_Subcategories"))

In [0]:
df_prod = (spark.read.format("csv")
         .option("header", True)
         .option("inferSchema", True)
         .load("abfss://bronze@sampeldatalake.dfs.core.windows.net/AdventureWorks_Products"))

In [0]:
df_ret = (spark.read.format("csv")
         .option("header", True)
         .option("inferSchema", True)
         .load("abfss://bronze@sampeldatalake.dfs.core.windows.net/AdventureWorks_Returns"))

In [0]:
df_Sales_2015 = (spark.read.format("csv")
         .option("header", True)
         .option("inferSchema", True)
         .load("abfss://bronze@sampeldatalake.dfs.core.windows.net/AdventureWorks_Sales_2015"))

In [0]:
df_Sales_2016 = (spark.read.format("csv")
         .option("header", True)
         .option("inferSchema", True)
         .load("abfss://bronze@sampeldatalake.dfs.core.windows.net/AdventureWorks_Sales_2016"))

In [0]:
df_Sales_2017 = (spark.read.format("csv")
         .option("header", True)
         .option("inferSchema", True)
         .load("abfss://bronze@sampeldatalake.dfs.core.windows.net/AdventureWorks_Sales_2017"))

In [0]:
df_Terr = (spark.read.format("csv")
         .option("header", True)
         .option("inferSchema", True)
         .load("abfss://bronze@sampeldatalake.dfs.core.windows.net/AdventureWorks_Territories"))

## TRANSFORMATIONS

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_cal = df_cal.withColumn('Month',month(col("Date")))\
    .withColumn('Year',year(col("Date")))
df_cal.display()


Date,Month,Year
2015-01-01,1,2015
2015-01-02,1,2015
2015-01-03,1,2015
2015-01-04,1,2015
2015-01-05,1,2015
2015-01-06,1,2015
2015-01-07,1,2015
2015-01-08,1,2015
2015-01-09,1,2015
2015-01-10,1,2015


In [0]:
df_cal.write.format("delta").mode("overwrite").save("abfss://silver@sampeldatalake.dfs.core.windows.net/AdventureWorks_Calender")

In [0]:
df_cal.write.format("parquet")\
            .mode("overwrite")\
            .option("path","abfss://silver@sampeldatalake.dfs.core.windows.net/AdventureWorks_Calender_parquet")\
            .save()

In [0]:
df_cust.show(5)

+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+-------------+--------------+------------+---------+
|CustomerKey|Prefix|FirstName|LastName| BirthDate|MaritalStatus|Gender|        EmailAddress|AnnualIncome|TotalChildren|EducationLevel|  Occupation|HomeOwner|
+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+-------------+--------------+------------+---------+
|      11000|   MR.|      JON|    YANG|1966-04-08|            M|     M|jon24@adventure-w...|    $90,000 |            2|     Bachelors|Professional|        Y|
|      11001|   MR.|   EUGENE|   HUANG|1965-05-14|            S|     M|eugene10@adventur...|    $60,000 |            3|     Bachelors|Professional|        N|
|      11002|   MR.|    RUBEN|  TORRES|1965-08-12|            M|     M|ruben35@adventure...|    $60,000 |            3|     Bachelors|Professional|        Y|
|      11003|   MS.|  CHRISTY|     ZHU|1968-02-15|  

In [0]:
df_cust.withColumn("fullname",concat(col("Prefix"),lit(' '),col("FirstName"),lit(' '), col("LastName"))).show(5)


+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+-------------+--------------+------------+---------+--------------------+
|CustomerKey|Prefix|FirstName|LastName| BirthDate|MaritalStatus|Gender|        EmailAddress|AnnualIncome|TotalChildren|EducationLevel|  Occupation|HomeOwner|            fullname|
+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+-------------+--------------+------------+---------+--------------------+
|      11000|   MR.|      JON|    YANG|1966-04-08|            M|     M|jon24@adventure-w...|    $90,000 |            2|     Bachelors|Professional|        Y|        MR. JON YANG|
|      11001|   MR.|   EUGENE|   HUANG|1965-05-14|            S|     M|eugene10@adventur...|    $60,000 |            3|     Bachelors|Professional|        N|    MR. EUGENE HUANG|
|      11002|   MR.|    RUBEN|  TORRES|1965-08-12|            M|     M|ruben35@adventure...|    $60,000 |

In [0]:
df_cust = df_cust.withColumn("FullName", concat_ws(' ','Prefix','FirstName','LastName'))
df_cust.show(5)

+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+-------------+--------------+------------+---------+--------------------+
|CustomerKey|Prefix|FirstName|LastName| BirthDate|MaritalStatus|Gender|        EmailAddress|AnnualIncome|TotalChildren|EducationLevel|  Occupation|HomeOwner|            FullName|
+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+-------------+--------------+------------+---------+--------------------+
|      11000|   MR.|      JON|    YANG|1966-04-08|            M|     M|jon24@adventure-w...|    $90,000 |            2|     Bachelors|Professional|        Y|        MR. JON YANG|
|      11001|   MR.|   EUGENE|   HUANG|1965-05-14|            S|     M|eugene10@adventur...|    $60,000 |            3|     Bachelors|Professional|        N|    MR. EUGENE HUANG|
|      11002|   MR.|    RUBEN|  TORRES|1965-08-12|            M|     M|ruben35@adventure...|    $60,000 |

In [0]:
df_cust.write.format("parquet").mode("overwrite").option("path","abfss://silver@sampeldatalake.dfs.core.windows.net/AdventureWorks_Customer").save()

In [0]:
df_prod_sub.show(5)
df_prod_sub.write.format("parquet").mode("overwrite").option("path","abfss://silver@sampeldatalake.dfs.core.windows.net/AdventureWorks_Product_Subcategory").save()

+---------------------+---------------+------------------+
|ProductSubcategoryKey|SubcategoryName|ProductCategoryKey|
+---------------------+---------------+------------------+
|                    1| Mountain Bikes|                 1|
|                    2|     Road Bikes|                 1|
|                    3|  Touring Bikes|                 1|
|                    4|     Handlebars|                 2|
|                    5|Bottom Brackets|                 2|
+---------------------+---------------+------------------+
only showing top 5 rows


In [0]:
df_prod.display()

ProductKey,ProductSubcategoryKey,ProductSKU,ProductName,ModelName,ProductDescription,ProductColor,ProductSize,ProductStyle,ProductCost,ProductPrice
214,31,HL-U509-R,"Sport-100 Helmet, Red",Sport-100,"Universal fit, well-vented, lightweight , snap-on visor.",Red,0,0,13.0863,34.99
215,31,HL-U509,"Sport-100 Helmet, Black",Sport-100,"Universal fit, well-vented, lightweight , snap-on visor.",Black,0,0,12.0278,33.6442
218,23,SO-B909-M,"Mountain Bike Socks, M",Mountain Bike Socks,Combination of natural and synthetic fibers stays dry and provides just the right cushioning.,White,M,U,3.3963,9.5
219,23,SO-B909-L,"Mountain Bike Socks, L",Mountain Bike Socks,Combination of natural and synthetic fibers stays dry and provides just the right cushioning.,White,L,U,3.3963,9.5
220,31,HL-U509-B,"Sport-100 Helmet, Blue",Sport-100,"Universal fit, well-vented, lightweight , snap-on visor.",Blue,0,0,12.0278,33.6442
223,19,CA-1098,AWC Logo Cap,Cycling Cap,Traditional style with a flip-up brim; one-size fits all.,Multi,0,U,5.7052,8.6442
226,21,LJ-0192-S,"Long-Sleeve Logo Jersey, S",Long-Sleeve Logo Jersey,Unisex long-sleeve AWC logo microfiber cycling jersey,Multi,S,U,31.7244,48.0673
229,21,LJ-0192-M,"Long-Sleeve Logo Jersey, M",Long-Sleeve Logo Jersey,Unisex long-sleeve AWC logo microfiber cycling jersey,Multi,M,U,31.7244,48.0673
232,21,LJ-0192-L,"Long-Sleeve Logo Jersey, L",Long-Sleeve Logo Jersey,Unisex long-sleeve AWC logo microfiber cycling jersey,Multi,L,U,31.7244,48.0673
235,21,LJ-0192-X,"Long-Sleeve Logo Jersey, XL",Long-Sleeve Logo Jersey,Unisex long-sleeve AWC logo microfiber cycling jersey,Multi,XL,U,31.7244,48.0673


In [0]:
df_prod = df_prod.withColumn("productSKU", split(col("ProductSKU"),'-')[0])\
                    .withColumn("productName", split(col("ProductName"),' ')[0])
df_prod.display()

ProductKey,ProductSubcategoryKey,productSKU,productName,ModelName,ProductDescription,ProductColor,ProductSize,ProductStyle,ProductCost,ProductPrice
214,31,HL,Sport-100,Sport-100,"Universal fit, well-vented, lightweight , snap-on visor.",Red,0,0,13.0863,34.99
215,31,HL,Sport-100,Sport-100,"Universal fit, well-vented, lightweight , snap-on visor.",Black,0,0,12.0278,33.6442
218,23,SO,Mountain,Mountain Bike Socks,Combination of natural and synthetic fibers stays dry and provides just the right cushioning.,White,M,U,3.3963,9.5
219,23,SO,Mountain,Mountain Bike Socks,Combination of natural and synthetic fibers stays dry and provides just the right cushioning.,White,L,U,3.3963,9.5
220,31,HL,Sport-100,Sport-100,"Universal fit, well-vented, lightweight , snap-on visor.",Blue,0,0,12.0278,33.6442
223,19,CA,AWC,Cycling Cap,Traditional style with a flip-up brim; one-size fits all.,Multi,0,U,5.7052,8.6442
226,21,LJ,Long-Sleeve,Long-Sleeve Logo Jersey,Unisex long-sleeve AWC logo microfiber cycling jersey,Multi,S,U,31.7244,48.0673
229,21,LJ,Long-Sleeve,Long-Sleeve Logo Jersey,Unisex long-sleeve AWC logo microfiber cycling jersey,Multi,M,U,31.7244,48.0673
232,21,LJ,Long-Sleeve,Long-Sleeve Logo Jersey,Unisex long-sleeve AWC logo microfiber cycling jersey,Multi,L,U,31.7244,48.0673
235,21,LJ,Long-Sleeve,Long-Sleeve Logo Jersey,Unisex long-sleeve AWC logo microfiber cycling jersey,Multi,XL,U,31.7244,48.0673


In [0]:
df_prod.write.format("parquet").mode("overwrite").option("path","abfss://silver@sampeldatalake.dfs.core.windows.net/AdventureWorks_Product").save()

In [0]:
df_ret.display()

ReturnDate,TerritoryKey,ProductKey,ReturnQuantity
2015-01-18,9,312,1
2015-01-18,10,310,1
2015-01-21,8,346,1
2015-01-22,4,311,1
2015-02-02,6,312,1
2015-02-15,1,312,1
2015-02-19,9,311,1
2015-02-24,8,314,1
2015-03-08,8,350,1
2015-03-13,9,350,1


In [0]:
df_ret.write.format("parquet").mode("overwrite").option("path","abfss://silver@sampeldatalake.dfs.core.windows.net/AdventureWorks_Return").save()
df_Sales_2015.write.format("parquet").mode("overwrite").option("path","abfss://silver@sampeldatalake.dfs.core.windows.net/AdventureWorks_Sales_2015").save()
df_Sales_2016.write.format("parquet").mode("overwrite").option("path","abfss://silver@sampeldatalake.dfs.core.windows.net/AdventureWorks_Sales_2016").save()
df_Sales_2017.write.format("parquet").mode("overwrite").option("path","abfss://silver@sampeldatalake.dfs.core.windows.net/AdventureWorks_Sales_2017").save()

In [0]:
df_Terr.write.format("parquet").mode("overwrite").option("path","abfss://silver@sampeldatalake.dfs.core.windows.net/AdventureWorks_Territory").save()
df_prod_catg.write.format("parquet").mode("overwrite").option("path", "abfss://silver@sampeldatalake.dfs.core.windows.net/AdventureWorks_ProductCategory").save()

In [0]:
df_Sales_2015.display()

OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
2015-01-01,2001-09-21,SO45080,332,14657,1,1,1
2015-01-01,2001-12-05,SO45079,312,29255,4,1,1
2015-01-01,2001-10-29,SO45082,350,11455,9,1,1
2015-01-01,2001-11-16,SO45081,338,26782,6,1,1
2015-01-02,2001-12-15,SO45083,312,14947,10,1,1
2015-01-02,2001-10-12,SO45084,310,29143,4,1,1
2015-01-02,2001-12-18,SO45086,314,18747,9,1,1
2015-01-02,2001-10-09,SO45085,312,18746,9,1,1
2015-01-03,2001-10-03,SO45093,312,18906,9,1,1
2015-01-03,2001-09-29,SO45090,310,29170,4,1,1


In [0]:
df_Sales_2015 = df_Sales_2015.withColumn("StockDate", to_timestamp("StockDate"))
df_Sales_2015.limit(5).display()

OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
2015-01-01,2001-09-21T00:00:00.000Z,SO45080,332,14657,1,1,1
2015-01-01,2001-12-05T00:00:00.000Z,SO45079,312,29255,4,1,1
2015-01-01,2001-10-29T00:00:00.000Z,SO45082,350,11455,9,1,1
2015-01-01,2001-11-16T00:00:00.000Z,SO45081,338,26782,6,1,1
2015-01-02,2001-12-15T00:00:00.000Z,SO45083,312,14947,10,1,1


In [0]:
df_Sales_2015 = df_Sales_2015.withColumn("OrderNumber", regexp_replace(col("OrderNumber"),"SO", ""))
df_Sales_2015.limit(5).display()

OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
2015-01-01,2001-09-21T00:00:00.000Z,45080,332,14657,1,1,1
2015-01-01,2001-12-05T00:00:00.000Z,45079,312,29255,4,1,1
2015-01-01,2001-10-29T00:00:00.000Z,45082,350,11455,9,1,1
2015-01-01,2001-11-16T00:00:00.000Z,45081,338,26782,6,1,1
2015-01-02,2001-12-15T00:00:00.000Z,45083,312,14947,10,1,1


In [0]:
df_Sales_2015 = df_Sales_2015.withColumn("ItemQuantity", col("OrderLineItem") * col('OrderQuantity'))
df_Sales_2015.limit(5).display()

OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity,total_orders,ItemQuantity
2015-01-01,2001-09-21T00:00:00.000Z,45080,332,14657,1,1,1,1,1
2015-01-01,2001-12-05T00:00:00.000Z,45079,312,29255,4,1,1,1,1
2015-01-01,2001-10-29T00:00:00.000Z,45082,350,11455,9,1,1,1,1
2015-01-01,2001-11-16T00:00:00.000Z,45081,338,26782,6,1,1,1,1
2015-01-02,2001-12-15T00:00:00.000Z,45083,312,14947,10,1,1,1,1


In [0]:
df_Sales_2015.write.format("parquet").mode("overwrite").option("path","abfss://silver@sampeldatalake.dfs.core.windows.net/AdventureWorks_Sales_2015").save()

##Sales Analysis

In [0]:
df_Sales_2015.groupBy("OrderDate").agg(count("OrderNumber")).alias("total_orders").display()


OrderDate,count(OrderNumber)
2015-01-09,4
2015-02-10,9
2015-02-13,6
2015-02-18,2
2015-03-09,6
2015-04-27,9
2015-05-02,5
2015-05-24,8
2015-06-12,12
2015-06-25,4


Databricks visualization. Run in Databricks to view.

In [0]:
df_Sales_2015.groupBy(year("OrderDate")).agg(count("OrderNumber")).alias("total_orders_per_year").display()

year(OrderDate),count(OrderNumber)
2015,2630


In [0]:
df_Terr.write.format("parquet").mode("overwrite").partitionBy("Continent").option("path","abfss://silver@sampeldatalake.dfs.core.windows.net/AdventureWorks_Territory").save()